In [ ]:
############################################################################################# check loss values ########################################################################################################################

import os
import torch
import torch.nn as nn
import nibabel as nib
import numpy as np
from scipy.ndimage import distance_transform_edt

############################################################
# Utility
############################################################

def get_tp_fp_fn_tn(pred, target, axes=None, mask=None, square=False):
    if axes is None:
        axes = list(range(2, len(pred.shape)))

    if mask is not None:
        pred = pred * mask
        target = target * mask

    tp = (pred * target).sum(dim=axes)
    fp = (pred * (1 - target)).sum(dim=axes)
    fn = ((1 - pred) * target).sum(dim=axes)
    tn = ((1 - pred) * (1 - target)).sum(dim=axes)

    return tp, fp, fn, tn


############################################################
# Losses
############################################################

class RobustCrossEntropyLoss(nn.CrossEntropyLoss):
    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if target.ndim == input.ndim:
            target = target[:, 0]
        return super().forward(input, target.long())


class SoftDiceLoss(nn.Module):
    def __init__(self, batch_dice=False, do_bg=True, smooth=1.0):
        super().__init__()
        self.batch_dice = batch_dice
        self.do_bg = do_bg
        self.smooth = smooth

    def forward(self, probs, target, loss_mask=None):
        shp = probs.shape
        axes = ([0] + list(range(2, len(shp)))) if self.batch_dice else list(range(2, len(shp)))

        tp, fp, fn, _ = get_tp_fp_fn_tn(probs, target, axes, loss_mask, False)

        dice = (2 * tp + self.smooth) / (2 * tp + fp + fn + self.smooth)

        if not self.do_bg:
            dice = dice[:, 1:]

        return -dice.mean()


class DC_and_CE_loss(nn.Module):
    def __init__(self, soft_dice_kwargs, ce_kwargs, weight_ce=1, weight_dice=1, weight_penalty=1):
        super().__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.weight_penalty = weight_penalty

        self.ce = RobustCrossEntropyLoss(**ce_kwargs)
        self.dc = SoftDiceLoss(**soft_dice_kwargs)

    def forward(self, logits, probs, target, distance_map=None):
        # CE uses logits
        ce_loss = self.ce(logits, target[:, 0]) if self.weight_ce else 0

        # Dice uses probabilities
        dice_loss = self.dc(probs, target) if self.weight_dice else 0

        # Penalty term
        penalty_loss = 0
        if distance_map is not None and self.weight_penalty > 0:
            prob_fg = probs[:, 1:2]  # predicted FG probability
            dist_norm = distance_map.to(prob_fg.device)
            penalty_loss = (prob_fg * dist_norm).mean()

        total = (
            self.weight_ce * ce_loss +
            self.weight_dice * dice_loss +
            self.weight_penalty * penalty_loss
        )

        return dice_loss, ce_loss, penalty_loss, total


############################################################
# Loaders
############################################################

def load_label(path):
    data = nib.load(path).get_fdata().astype(np.int64)
    t = torch.tensor(data)[None, None]      # (1,1,X,Y,Z)
    return t


def load_probs_from_npz(npz_path):
    data = np.load(npz_path)
    probs = data["probabilities"]           # (C,X,Y,Z)
    probs = torch.tensor(probs)[None]       # → (1,C,X,Y,Z)
    print(probs.min(),probs.max(),probs.shape)
    print(probs[:, 0:1].sum(), probs[:, 0:1].max(), probs[:, 0:1].min())
    print(probs[:, 1:2].sum(), probs[:, 1:2].max(), probs[:, 1:2].min()) 
    return probs     # → (1,C,Z,Y,X)


def compute_distance_map(label_tensor):
    """
    Compute distance map from GT (1,1,Z,Y,X)
    distance_transform_edt applied to background (label==0)
    then normalized with log1p
    """
    B, C, Z, Y, X = label_tensor.shape
    dist_map = torch.zeros_like(label_tensor, dtype=torch.float32)

    for b in range(B):
        for i in range(C):
            # distance_transform_edt on background voxels
            dist = distance_transform_edt((label_tensor[b, i].cpu().numpy() == 0))
            dist_map[b, i] = torch.tensor(np.log1p(dist), dtype=torch.float32)
    print(dist_map.min(),dist_map.max(),dist_map.shape)

    return dist_map


############################################################
# MAIN
############################################################

folder_A = "/data/colon_cancer/CC_Detection/nnUNet_results/Dataset105_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation"
folder_B = "/data/colon_cancer/CC_Detection/pp_data/Dataset105_CC/gt_segmentations"

loss_fn = DC_and_CE_loss(
    soft_dice_kwargs={"batch_dice": False, "do_bg": False, "smooth":  1e-5},
    ce_kwargs={},
    weight_ce=1,
    weight_dice=1,
    weight_penalty=1,
)

uid_list = ["1.npz"]

for uid in uid_list:

    pred_path = os.path.join(folder_A, uid)
    label_path = os.path.join(folder_B, uid.replace(".npz", ".nii.gz"))

    if not os.path.exists(label_path):
        print("Missing GT", uid)
        continue

    # Load probabilities
    probs = load_probs_from_npz(pred_path)      # (1,C,Z,Y,X)

    # logits for CE
    logits = torch.log(probs + 1e-8)

    # Load GT
    label = load_label(label_path)              # (1,1,X,Y,Z)
    label = label.permute(0,1,4,2,3)           # → (1,1,Z,Y,X)

    # Compute distance map
    dist = compute_distance_map(label)

    # Compute losses
    dice_loss, ce_loss, penalty_loss, total = loss_fn(logits, probs, label, dist)

    print("\nCase:", uid)
    print("  Dice loss:", float(dice_loss))
    print("  CE loss:", float(ce_loss))
    print("  Penalty loss:", float(penalty_loss))
    print("  Total:", float(total))


In [ ]:
############################################################# check distance map visualization ########################################################################################################################

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets

# ===========================
# Distance map computation
# ===========================

def compute_distance_maps(mask):
    mask_bool = mask.astype(bool)
    dist_in = distance_transform_edt(mask_bool)
    dist_out = distance_transform_edt(~mask_bool)
    signed_dist = dist_in - dist_out
    return dist_in, dist_out, signed_dist

def build_distance_overlay(distance_slice, alpha=0.35):
    """RGBA overlay for distance map (signed)."""
    # Normalize for visualization only
    normed = (distance_slice - distance_slice.min()) / (distance_slice.max() - distance_slice.min() + 1e-6)
    overlay = plt.cm.jet(normed)
    overlay[..., 3] = alpha
    return overlay

# ===========================
# Interactive axial visualization
# ===========================

def visualize_distance_case(ct_volume, mask_volume, axis="axial"):
    dist_in, dist_out, signed_dist = compute_distance_maps(mask_volume)
    
    n_slices = ct_volume.shape[0] if axis=="axial" else ct_volume.shape[1] if axis=="coronal" else ct_volume.shape[2]
    
    def plot_slice(idx):
        if axis=="axial":
            ct_slice = ct_volume[idx,:,:]
            mask_slice = mask_volume[idx,:,:]
            dist_in_slice = dist_in[idx,:,:]
            dist_out_slice = dist_out[idx,:,:]
            signed_slice = signed_dist[idx,:,:]
        elif axis=="coronal":
            ct_slice = ct_volume[:,idx,:]
            mask_slice = mask_volume[:,idx,:]
            dist_in_slice = dist_in[:,idx,:]
            dist_out_slice = dist_out[:,idx,:]
            signed_slice = signed_dist[:,idx,:]
        elif axis=="sagittal":
            ct_slice = ct_volume[:,:,idx]
            mask_slice = mask_volume[:,:,idx]
            dist_in_slice = dist_in[:,:,idx]
            dist_out_slice = dist_out[:,:,idx]
            signed_slice = signed_dist[:,:,idx]
        else:
            raise ValueError(f"Invalid axis {axis}")

        # Print statistics
        print(f"Slice {idx}: dist_in max {dist_in_slice.max():.2f}, dist_out max {dist_out_slice.max():.2f}, signed_dist min {signed_slice.min():.2f}, max {signed_slice.max():.2f}")

        mask_overlay = np.zeros_like(mask_slice)
        mask_overlay[mask_slice>0] = 1
        signed_overlay = build_distance_overlay(signed_slice)

        plt.figure(figsize=(20,6))
        
        # CT
        plt.subplot(1,5,1)
        plt.imshow(ct_slice, cmap="gray", origin="lower")
        plt.title("CT")
        plt.axis("off")
        
        # Mask overlay
        plt.subplot(1,5,2)
        plt.imshow(ct_slice, cmap="gray", origin="lower")
        plt.imshow(mask_overlay, cmap=ListedColormap([[0,0,0,0],[1,0,0,0.35]]), origin="lower")
        plt.title("Mask overlay")
        plt.axis("off")
        
        # dist_in
        plt.subplot(1,5,3)
        plt.imshow(dist_in_slice, cmap="jet", origin="lower")
        plt.title("dist_in")
        plt.colorbar()
        plt.axis("off")
        
        # dist_out
        plt.subplot(1,5,4)
        plt.imshow(dist_out_slice, cmap="jet", origin="lower")
        plt.title("dist_out")
        plt.colorbar()
        plt.axis("off")
        
        # signed distance overlay
        plt.subplot(1,5,5)
        plt.imshow(ct_slice, cmap="gray", origin="lower")
        plt.imshow(signed_overlay, origin="lower")
        plt.title("Signed distance overlay")
        plt.axis("off")

        plt.tight_layout()
        plt.show()
    
    slice_slider = widgets.IntSlider(value=n_slices//2, min=0, max=n_slices-1, step=1, description="Slice", continuous_update=False)
    widgets.interact(plot_slice, idx=slice_slider)

ct_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr/4_0000.nii.gz"
mask_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr/4.nii.gz"

ct = load_volume(ct_path)
mask = load_volume(mask_path)

visualize_distance_case(ct, mask, axis="sagittal")


In [ ]:
########################################################### visualize distance maps for loss computation ########################################################################################################################


from pathlib import Path

# ------------ LOADING -------------------------------------------------

def load_b2nd(path):
    import blosc2
    schunk = blosc2.open(path, mode="r")
    arr = schunk[:][0]
    return arr

def load_volume(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
    elif path.suffix in [".nii", ".gz"]:
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")

# ------------ DISTANCES -----------------------------------------------

def compute_distance_maps(mask):
    mask_bool = mask.astype(bool)
    dist_in = distance_transform_edt(mask_bool)
    dist_out = distance_transform_edt(~mask_bool)
    signed_dist = dist_in - dist_out
    return dist_in, dist_out, signed_dist

def normalize_global(arr):
    arr_min, arr_max = arr.min(), arr.max()
    return (arr - arr_min) / (arr_max - arr_min + 1e-6)

# ------------ VISUALIZATION -------------------------------------------

def visualize_distances_extended(ct_volume, mask_volume, axis="axial"):
    """
    Adds visualization of:
        - original dist_out
        - normalized dist_out
        - log(1 + dist_out)
    """

    dist_in, dist_out, signed_dist = compute_distance_maps(mask_volume)

    # Prepare additional maps
    dist_out_norm = normalize_global(dist_out)
    log1p_dist = np.log1p(dist_out)
   

    # Axis selection
    if axis == "axial":
        n_slices = ct_volume.shape[0]
    elif axis == "coronal":
        n_slices = ct_volume.shape[1]
    elif axis == "sagittal":
        n_slices = ct_volume.shape[2]
    else:
        raise ValueError("axis must be axial/coronal/sagittal")

    def extract_slice(vol, idx):
        if axis == "axial":
            return vol[idx]
        elif axis == "coronal":
            return vol[:, idx, :]
        else:
            return vol[:, :, idx]

    def plot_slice(idx):

        # Extract slices
        ct_slice = extract_slice(ct_volume, idx)
        mask_slice = extract_slice(mask_volume, idx)
        dist_out_slice = extract_slice(dist_out, idx)
        dist_out_norm_slice = extract_slice(dist_out_norm, idx)
        log1p_slice = extract_slice(log1p_dist, idx)
       

        # Print numerical stats
        print("---- Slice", idx, "----")
        print("Original dist_out:", dist_out_slice.min(), dist_out_slice.max())
        print("Normalized dist_out:", dist_out_norm_slice.min(), dist_out_norm_slice.max())
        print("log(1+dist_out):", log1p_slice.min(), log1p_slice.max())
        print()

        plt.figure(figsize=(26, 10))

        # 1) CT + mask overlay
        plt.subplot(1, 6, 1)
        plt.imshow(ct_slice, cmap="gray", origin="lower")
        mask_rgb = np.zeros((*mask_slice.shape, 4))
        mask_rgb[mask_slice > 0] = [1, 0, 0, 0.4]
        plt.imshow(mask_rgb, origin='lower')
        plt.title("CT + Mask")
        plt.axis("off")

        # 2) Original dist_out
        plt.subplot(1, 6, 2)
        plt.imshow(dist_out_slice, cmap="jet", origin="lower")
        plt.title("dist_out (original)")
        plt.colorbar()
        plt.axis("off")

        # 3) dist_out normalized
        plt.subplot(1, 6, 3)
        plt.imshow(dist_out_norm_slice, cmap="jet", origin="lower")
        plt.title("dist_out_norm")
        plt.colorbar()
        plt.axis("off")

        # 4) log(1 + dist_out)
        plt.subplot(1, 6, 4)
        plt.imshow(log1p_slice, cmap="jet", origin="lower")
        plt.title("log(1 + dist_out)")
        plt.colorbar()
        plt.axis("off")


        plt.tight_layout()
        plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices // 2,
        min=0,
        max=n_slices - 1,
        step=1,
        description="Slice",
        continuous_update=False
    )

    widgets.interact(plot_slice, idx=slice_slider)


# ------------ EXAMPLE USAGE -------------------------------------------

ct_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr/123_0000.nii.gz"
mask_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTr/123.nii.gz"

ct = load_volume(ct_path)
mask = load_volume(mask_path)

visualize_distances_extended(ct, mask, axis="sagittal")


In [ ]:
######################################################################## check training samples #################################################

import sys
from IPython.display import display
import torch 
import torchio as tio
import os
from pathlib import Path 
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
# ----------------------------
# Load dataset
# ----------------------------
project_root = Path("/data/benchaaben/ColonCancerDetection/classifier")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data import basedataset
from torch.utils.data import DataLoader
patch_shape = (32, 156, 156)

dataset = basedataset(dataset_name= "Dataset109_CC", patch_size=patch_shape, split="train", path_root="/data/colon_cancer/CC_Detection",dual_input=True,return_full_image=True)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

sample = next(iter(loader))
#sample=dataset.get_item_by_uid("1")
print(sample["source"].shape)

image = sample["source"][0,0] # shape [ D, H, W]
label = sample["source"][0,1] # shape [ D, H, W]



subject = tio.Subject(
    image=tio.ScalarImage(tensor=image.unsqueeze(0)),  
    label=tio.LabelMap(tensor=label.unsqueeze(0))
)

transform = tio.Compose([
    tio.RandomFlip(axes=('LR', 'AP', 'IS'), flip_probability=0.4),
    tio.RandomAffine(degrees=5)
])

# Apply transforms
transformed = transform(subject)
transformed_patch = transformed['image'].data.squeeze(0)      
transformed_label = transformed['label'].data.squeeze(0)  


# ----------------------------
# Visualization function
# ----------------------------
def view_transformed(slice_idx):
    fig, axes = plt.subplots(1, 1, figsize=(10,5))
    
    # Original
    axes.imshow(image[slice_idx, :, : ], cmap='gray')
    axes.imshow(label[slice_idx, :, :], cmap='Reds', alpha=0.2)
    print(label[slice_idx,:,:].sum())
    axes.set_title(f'Original Slice {slice_idx}')
    axes.axis('off')
    """
    # Apply transform
    axes[1].imshow(transformed_patch[slice_idx, :, :], cmap='gray')
    axes[1].imshow(transformed_label[slice_idx, :, :], cmap='Reds', alpha=0.4)
    axes[1].set_title(f'Transformed Slice {slice_idx}')
    axes[1].axis('off')
    """
    #os.makedirs(f"/data/benchaaben/classifier/{uid}", exist_ok=True)
    #save_path = os.path.join(f"/data/benchaaben/classifier/{uid}/pred_{slice_idx}.png")
    #plt.savefig(save_path, bbox_inches='tight', dpi=150)
    #plt.show() 
d = image.shape[0]
slider = widgets.IntSlider(min=0, max=d-1, step=1, value=d//2, description='Slice')
widgets.interact(view_transformed, slice_idx=slider)


[BaseDataset] Loaded 600 subjects for split='train'
(119, 81, 97) (119, 81, 97)
torch.Size([1, 2, 119, 81, 97])


interactive(children=(IntSlider(value=59, description='Slice', max=118), Output()), _dom_classes=('widget-inte…

<function __main__.view_transformed(slice_idx)>

In [16]:
############################### visualize all labels: total_segmentatot, bowel wall thickening #############################

import sys
from pathlib import Path
import ipywidgets as widgets


from matplotlib.colors import to_rgba
from matplotlib import cm
from matplotlib.colors import BoundaryNorm
sys.path.append(str(Path.cwd().parent))
from utils.io_utils import load_nifti
from matplotlib.patches import Patch
import os
# --- config ---
#images_dir = Path("/data/colon_cancer/CC_update/image")
images_dir = Path("/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr")

LABEL_MAPS = {
    "tissue_4_types": {
        "background": 0,
        "subcutaneous_fat": 1,
        "torso_fat": 2,
        "skeletal_muscle": 3,
        "intermuscular_fat": 4,
    },
    "colon": {
        "background": 0,
        "colon": 20,
    },
    "Bowel_thickening": {
        "background": 0,
        "bowel_thickening": 1,
    },
}

# --- Input label directories with their corresponding maps ---
LABEL_DIRS_AND_MAPS = [
    #(Path("/data/benchaaben/colon_seg/outputs/tissue_4_types"), LABEL_MAPS["tissue_4_types"]),
    #(Path("/data/colon_cancer/totalseg/total"), LABEL_MAPS["colon"]),
    (Path("/data/benchaaben/colon_seg/outputs/total"), LABEL_MAPS["colon"]),
    #(Path("/data/colon_cancer/totalseg/total"), LABEL_MAPS["colon"]),
    #(Path("/data/colon_cancer/Task101_Colon/raw_splitted/labelsTs"), LABEL_MAPS["Bowel_thickening"]),
    
    #(Path("/data/colon_cancer/CC_update/segs"), LABEL_MAPS["Bowel_thickening"])
]

WINDOW_LEVEL = 50
WINDOW_WIDTH = 350

OVERLAY_ALPHA = 0.30  # slightly higher for clearer overlays

# High-contrast palette (Glasbey-like). Will cycle if you have more labels.
HIGH_CONTRAST_COLORS = [
    "#0000FF", "#FF0000", "#00FF00", "#FF00FF", "#00FFFF", "#FFFF00", "#000000", "#FF8000",
    "#8000FF", "#0080FF", "#80FF00", "#FF0080", "#00FF80", "#808000", "#008000", "#800000",
    "#000080", "#804000", "#408000", "#008040", "#400080", "#804080", "#408080", "#808040",
    "#FF8080", "#80FF80", "#8080FF", "#FF80FF", "#80FFFF", "#FFFF80", "#404040", "#C00000",
]

# --- helpers ---
def window_ct_hu(ct_hu: np.ndarray, level: float = WINDOW_LEVEL, width: float = WINDOW_WIDTH) -> np.ndarray:
    lower = level - width / 2.0
    upper = level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    ct_norm = (ct_clipped - lower) / (upper - lower + 1e-6)
    return ct_norm


def get_image_label_pairs(indices=None, n_random=2):
    rng = np.random.default_rng(0)
    all_cts = sorted(images_dir.glob("*.nii.gz"))
    all_indices = [int(f.name.split("_")[0]) for f in all_cts]
    if indices is None:
        chosen = list(rng.choice(all_indices, size=n_random, replace=False))
    else:
        chosen = indices

    pairs = []
    for idx in chosen:
        ct_file = images_dir / f"{idx:03d}_0000.nii.gz"
        label_files = []
        for label_dir, _ in LABEL_DIRS_AND_MAPS:
            lf = label_dir / f"{idx:03d}.nii.gz"
            label_files.append(lf if lf.exists() else None)
        pairs.append((ct_file, label_files))
    return pairs


def build_distinct_cmap(n_labels: int, alpha: float = OVERLAY_ALPHA) -> ListedColormap:
    # Index 0 is background (transparent). Others draw from HIGH_CONTRAST_COLORS, then HSV fallback.
    colors = np.zeros((n_labels, 4), dtype=float)
    if n_labels == 0:
        return ListedColormap(colors)

    colors[0] = [0, 0, 0, 0]

    num_needed = max(0, n_labels - 1)
    base_rgba = []

    # Use high-contrast palette first (cycled if needed)
    if num_needed > 0:
        if num_needed <= len(HIGH_CONTRAST_COLORS):
            picks = HIGH_CONTRAST_COLORS[:num_needed]
        else:
            # cycle through list, then add HSV distinct hues for overflow
            cycles = [HIGH_CONTRAST_COLORS[i % len(HIGH_CONTRAST_COLORS)] for i in range(num_needed)]
            overflow = num_needed - len(HIGH_CONTRAST_COLORS)
            if overflow > 0:
                hsv_more = cm.hsv(np.linspace(0, 1, overflow, endpoint=False))[:, :3]
                cycles[-overflow:] = [tuple(rgb) for rgb in hsv_more]
            picks = cycles

        for p in picks:
            try:
                base_rgba.append(to_rgba(p, alpha=None))
            except Exception:
                base_rgba.append((0.5, 0.5, 0.5, 1.0))

    if num_needed > 0:
        colors[1:1+num_needed, :3] = np.array(base_rgba)[:, :3]
        colors[1:1+num_needed, 3] = alpha

    return ListedColormap(colors)

def show_interactive_pair(ct_path: Path, label_paths):
    # --- Load CT ---
    ct, _, _, _ = load_nifti(ct_path)
    print(ct.shape)
    # --- Build globally unique remapped labels across all provided label files ---
    # We assign new ids like: 1..K (0 is background). Each source map contributes len(non-bg) ids.
    all_remapped_labels = []
    id_to_name = {}  # global_id -> readable name
    current_id = 1

    for lp, (_, label_map) in zip(label_paths, LABEL_DIRS_AND_MAPS):
        if lp is None or not lp.exists():
            continue

        label_data, _, _, _ = load_nifti(lp)
        label_data = label_data.astype(np.int32)
        

        # Create a remapped array for this file
        remapped = np.zeros_like(label_data, dtype=np.int32)

        # Stable order: iterate label_map keys except background
        non_bg_items = [(name, val) for name, val in label_map.items() if val != 0]

        for name, original_val in non_bg_items:
            remapped[label_data == original_val] = current_id
            id_to_name[current_id] = name
            current_id += 1

        all_remapped_labels.append(remapped)

    n_total_labels = current_id  # includes 0 (background) up to last assigned id
    global_cmap = build_distinct_cmap(n_total_labels)

    # --- Interactive plotting ---
    def plot_slice(slice_idx: int):
        plt.figure(figsize=(6, 6))
        ct_ax = ct[:, :, slice_idx]
        ct_img = window_ct_hu(ct_ax, WINDOW_LEVEL, WINDOW_WIDTH)
        plt.imshow(ct_img.T, cmap="gray", origin="lower")

        # Combine overlays; later label files take precedence where overlapping
        combined_overlay = np.zeros_like(ct_ax, dtype=np.int32)
        for remapped in all_remapped_labels:
            lab_ax = remapped[:, :, slice_idx]
            combined_overlay = np.where(lab_ax > 0, lab_ax, combined_overlay)
        norm = BoundaryNorm(boundaries=np.arange(n_total_labels+1)-0.5, ncolors=n_total_labels)
        # Draw overlay once with the global colormap
        if n_total_labels > 1:
            plt.imshow(combined_overlay.T, cmap=global_cmap, norm=norm, origin="lower", interpolation="nearest")

        # Build legend only for labels present in this slice
        present_ids = set(np.unique(combined_overlay)) - {0}
        legend_elements = []
        for gid in sorted(present_ids):
            name = id_to_name.get(int(gid), f"Label {int(gid)}")
            rgba = global_cmap(norm(int(gid)))  # use same norm as imshow
            legend_elements.append(Patch(facecolor=rgba, edgecolor='k', label=name))

        plt.axis("off")
        plt.title(f"{ct_path.stem} - slice {slice_idx}")
        #os.makedirs(f"/data/benchaaben/classifier/{ct_path.stem}", exist_ok=True)
        #save_path = os.path.join(f"/data/benchaaben/classifier/{ct_path.stem}/pred_{slice_idx}.png")
        #plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        if legend_elements:
            plt.figure(figsize=(max(4, len(legend_elements)), 1.2))
            plt.legend(handles=legend_elements, loc='center', ncol=len(legend_elements), frameon=False)
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    slice_slider = widgets.IntSlider(
        value=ct.shape[2] // 2,
        min=0,
        max=ct.shape[2] - 1,
        step=1,
        description=f"Slice {ct_path.stem}:",
        continuous_update=False,
    )
    display(widgets.interact(plot_slice, slice_idx=slice_slider))


def show_images(indices=None, n_random=2):
    pairs = get_image_label_pairs(indices=indices, n_random=n_random)
    for ct_path, label_paths in pairs:
        show_interactive_pair(ct_path, label_paths)


# --- Examples ---
# show_images(n_random=2)
show_images(indices=[2])
#show_images(indices=[18,29,300])

ModuleNotFoundError: No module named 'utils'

In [ ]:
#################################################### test cropping function #####################################################################

import numpy as np
import ipywidgets as widgets
from ipywidgets import interact
from matplotlib.colors import ListedColormap
from typing import Tuple, List, Sequence

# --------------------------
# Helper functions (same as before)
# --------------------------

def voxels_from_mm(spacing, margin_mm: Tuple[float, float, float]):
    return tuple([int(np.ceil(mm / sp)) for mm, sp in zip(margin_mm, spacing)])

def get_bbox_from_mask_with_margin(mask: np.ndarray, margin_vox: Tuple[int, int, int]) -> List[Tuple[int, int]]:
    nonzero = np.where(mask > 0)
    if len(nonzero[0]) == 0:
        return [(0, mask.shape[0]), (0, mask.shape[1]), (0, mask.shape[2])]
    bbox = []
    for d in range(3):
        start = max(int(np.min(nonzero[d])) - margin_vox[d], 0)
        end = min(int(np.max(nonzero[d])) + margin_vox[d] + 1, mask.shape[d])
        bbox.append((start, end))
    return bbox

def crop_to_bbox_no_channels(image, bbox: Sequence[Sequence[int]]):
    resizer = tuple(slice(start, end) for start, end in bbox)
    return image[resizer]

def crop_to_label_region(data: np.ndarray,
                         crop_mask :np.ndarray,
                         seg: np.ndarray,
                         spacing: Tuple[float, float, float],
                         margin_min: float = 15.0) -> Tuple[np.ndarray, np.ndarray, List[Tuple[int, int]]]:
    margin_vox = voxels_from_mm(spacing, (margin_min, margin_min, margin_min))
    bbox = get_bbox_from_mask_with_margin(crop_mask, margin_vox)
    data_cropped = crop_to_bbox_no_channels(data, bbox) if data.ndim==3 else None
    seg_cropped = crop_to_bbox_no_channels(seg, bbox)
    return data_cropped, seg_cropped, bbox

# --------------------------
# Main function with red label overlay
# --------------------------

def load_crop_visualize(uid: str,
                        image_path_template: str,
                        label_path_template: str,
                        spacing: Tuple[float,float,float]=(1.0,1.0,1.0),
                        margin_mm: float = 15.0,
                        window_center: float = 50,
                        window_width: float = 350,
                        overlay_alpha: float = 0.2):
    # Load image & label
    img_path = image_path_template.format(uid=uid)
    lbl_path = label_path_template.format(uid=uid)
    img = nib.load(img_path).get_fdata()
    lbl = nib.load(lbl_path).get_fdata().astype(np.uint8)

    # Crop to label region
    img_crop, lbl_crop, bbox = crop_to_label_region(img, lbl, lbl, spacing, margin_min=margin_mm)
    print(f"Cropped bounding box (voxels): {bbox}")
    print(f"Cropped image shape: {img_crop.shape}, Cropped label shape: {lbl_crop.shape}")

    # Apply CT windowing
    lower = window_center - window_width / 2
    upper = window_center + window_width / 2
    img_crop = np.clip(img_crop, lower, upper)
    img_crop = (img_crop - lower) / window_width  # normalize 0-1

    # Prepare red colormap for label
    red_cmap = ListedColormap([[0,0,0,0],[1,0,0,1]])  # index 0 transparent, 1 red

    # Interactive slice viewer
    def show_slice(idx):
        plt.figure(figsize=(6,6))
        plt.imshow(img_crop[:, :, idx], cmap='gray')
        plt.imshow(lbl_crop[:, :, idx], cmap=red_cmap, alpha=overlay_alpha)
        plt.title(f"Slice {idx}")
        plt.axis('off')
        plt.show()
    
    interact(show_slice, idx=widgets.IntSlider(min=0, max=img_crop.shape[2]-1, value=img_crop.shape[2]//2))

# --------------------------
# Example usage
# --------------------------

uid = 724
image_template = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTs/{uid}_0000.nii.gz"
label_template = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs/{uid}.nii.gz"

load_crop_visualize(uid, image_template, label_template,
                    spacing=(1.0,1.0,1.0),
                    margin_mm=15,
                    window_center=50,
                    window_width=350,
                    overlay_alpha=0.2)


In [ ]:
import torch
from torchmetrics import Accuracy, Precision, Recall

# =========================================================
# 1. Dummy data (same as yours)
# =========================================================

logits = torch.tensor([
    [ 1.0762, -1.2031],
        [ 5.2656, -5.6133],
        [ 5.5078, -5.8438],
        [ 6.8906, -6.6719],
        [ 8.0938, -8.1797],
        [ 2.3340, -2.4668],
        [ 3.2910, -3.2363],
        [ 3.3633, -3.3984]
])

target = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1])
num_classes = 2

preds = torch.argmax(logits, dim=1)

# =========================================================
# 2. Metrics
# =========================================================

acc = Accuracy(task="multiclass", num_classes=num_classes)

precision_micro = Precision(
    task="multiclass",
    num_classes=num_classes,
    average="micro"
)

precision_macro = Precision(
    task="multiclass",
    num_classes=num_classes,
    average="macro"
)

precision_per_class = Precision(
    task="multiclass",
    num_classes=num_classes,
    average=None
)

recall_micro = Recall(
    task="multiclass",
    num_classes=num_classes,
    average="micro"
)

recall_macro = Recall(
    task="multiclass",
    num_classes=num_classes,
    average="macro"
)

recall_per_class = Recall(
    task="multiclass",
    num_classes=num_classes,
    average=None
)

# =========================================================
# 3. Update metrics
# =========================================================

acc.update(logits, target)

precision_micro.update(logits, target)
precision_macro.update(logits, target)
precision_per_class.update(logits, target)

recall_micro.update(logits, target)
recall_macro.update(logits, target)
recall_per_class.update(logits, target)

# =========================================================
# 4. Print results
# =========================================================

print("Predicted classes:", preds.tolist())
print("Targets          :", target.tolist())
print()

print(f"Accuracy              : {acc.compute().item():.4f}")
print()

print(f"Precision (micro)     : {precision_micro.compute().item():.4f}")
print(f"Recall    (micro)     : {recall_micro.compute().item():.4f}")
print("→ micro precision = micro recall = accuracy")
print()

print(f"Precision (macro)     : {precision_macro.compute().item():.4f}")
print(f"Recall (macro)     : {recall_macro.compute().item():.4f}")
